# Phase 3 - Step 4: Lesion Segmentation (10 1 YOLOv11 lesion)

Unlike U-Net (which does Semantic Segmentation pixel-by-pixel), YOLOv11 finds *objects* and draws polygons around them. This makes it incredibly powerful for finding tiny dots (Microaneurysms) because it has specialized Feature Pyramids for small objects.


## 1. Setup & Imports


In [ ]:
!pip install ultralytics -q

In [ ]:
!pip install ultralytics -q
import os
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from ultralytics import YOLO

# Import our evaluation metrics for a fair comparison later



In [ ]:
"""
Retinal DR Detection — Lesion Segmentation Dataset
=====================================================
Multi-label dataset for simultaneous segmentation of 5 lesion types.

Key difference from VesselDataset (src/dataset.py):
  - Vessel segmentation = SINGLE binary mask  → model outputs (B, 1, H, W)
  - Lesion segmentation = FIVE  binary masks  → model outputs (B, 5, H, W)

Each image produces a stacked mask tensor of shape (5, H, W) where:
  Channel 0 → MA  (Microaneurysms)        — earliest sign of DR
  Channel 1 → HE  (Hemorrhages)           — blood leaking from vessels
  Channel 2 → EX  (Hard Exudates)         — lipid deposits, sign of macular edema
  Channel 3 → OD  (Optic Disc)            — anatomical landmark for spatial reference
  Channel 4 → CW  (Cotton Wool spots)     — nerve fiber infarcts, severe ischemia

Albumentations automatically applies the SAME spatial transform (flip,
rotate, distort) to ALL mask channels, ensuring geometric consistency.
"""

import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2


# ============================================================
# The 5 lesion types — this is the SINGLE source of truth
# ============================================================
LESION_TYPES = ['MA', 'HE', 'EX', 'OD', 'CW']
NUM_LESION_CLASSES = len(LESION_TYPES)


class LesionDataset(Dataset):
    """
    Multi-label retinal lesion segmentation dataset.

    For each fundus image it loads 5 binary masks (one per lesion type)
    and stacks them into a single (5, H, W) tensor.  If a mask file
    doesn't exist for a particular lesion type, a zero-mask (= "no
    lesion present") is used — this is the correct ground truth.

    Args:
        csv_file     : Path to CSV with 'img_id' and 'img_path' columns.
        base_dir     : Root directory of the processed dataset
                       (e.g. 'dataset_stage1_segmentation_processed').
        transform    : Albumentations Compose pipeline (from get_*_transforms).
        lesion_mask_dir : Sub-folder containing per-type mask folders
                         (default 'lesion_masks').
        lesion_types : Which lesion types to load (default LESION_TYPES).
    """

    def __init__(self, csv_file, base_dir, transform=None,
                 lesion_mask_dir='lesion_masks',
                 lesion_types=None):
        self.df = pd.read_csv(csv_file)

        # Keep only rows that have at least one lesion mask
        if 'has_lesion' in self.df.columns:
            self.df = self.df[self.df['has_lesion'] == True].reset_index(drop=True)

        self.base_dir = base_dir
        self.lesion_mask_dir = lesion_mask_dir
        self.transform = transform
        self.lesion_types = lesion_types or LESION_TYPES

    def __len__(self):
        return len(self.df)

    # ----------------------------------------------------------
    def _load_single_mask(self, img_id, lesion_type):
        """
        Attempt to load one lesion mask from disk.

        Path convention (set during preprocessing):
            {base_dir}/lesion_masks/{lesion_type}/{img_id}
            e.g.  dataset_processed/lesion_masks/MA/IDRID_train_IDRiD_01.png

        Returns:
            np.ndarray (H, W) float32 with values 0.0 / 1.0, or None.
        """
        mask_path = os.path.join(
            self.base_dir, self.lesion_mask_dir, lesion_type, img_id
        )
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is not None:
                return (mask > 127).astype(np.float32)
        return None

    # ----------------------------------------------------------
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['img_id']               # e.g. "IDRID_train_IDRiD_01.png"

        # ---- 1. Load RGB image ----
        img_path = os.path.join(self.base_dir, row['img_path'])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w = image.shape[:2]

        # ---- 2. Load & stack all 5 lesion masks → (H, W, 5) ----
        #   Albumentations treats a 3-D mask array as a multi-channel
        #   mask and applies the same geometric transform to every channel.
        mask_stack = np.zeros((h, w, len(self.lesion_types)), dtype=np.float32)
        for i, lt in enumerate(self.lesion_types):
            m = self._load_single_mask(img_id, lt)
            if m is not None:
                mask_stack[:, :, i] = m

        # ---- 3. Apply augmentations ----
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask_stack)
            image = augmented['image']       # Tensor (3, H, W) after ToTensorV2
            mask_stack = augmented['mask']    # Tensor (5, H, W) after ToTensorV2

        # ---- 4. Ensure correct tensor format: (C, H, W) ----
        #   ToTensorV2 in newer albumentations does NOT auto-transpose
        #   multi-channel masks, so we handle it explicitly.
        if isinstance(mask_stack, np.ndarray):
            # No transform applied — convert manually
            mask_tensor = torch.from_numpy(
                mask_stack.transpose(2, 0, 1)  # (H, W, 5) → (5, H, W)
            ).float()
        elif isinstance(mask_stack, torch.Tensor):
            if mask_stack.dim() == 3 and mask_stack.shape[-1] == len(self.lesion_types):
                # Shape is (H, W, 5) — need to permute to (5, H, W)
                mask_tensor = mask_stack.permute(2, 0, 1).float()
            else:
                mask_tensor = mask_stack.float()
        else:
            mask_tensor = torch.as_tensor(mask_stack).float()

        return image, mask_tensor


# ============================================================
# Augmentation Pipelines (same as vessel, reused here)
# ============================================================

def get_train_transforms(img_size=512):
    """
    Training augmentation pipeline.

    Geometric transforms handle fundus image variability:
      - Different camera angles       → flips & rotations
      - Different fields of view      → scale variations
      - Patient movement              → shift & distortion

    Color transforms handle:
      - Different fundus camera brands → brightness/contrast variation
      - Different illumination         → hue/saturation shifts
    """
    return A.Compose([
        A.Resize(img_size, img_size),

        # === Geometric Augmentations ===
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent={'x': (-0.0625, 0.0625), 'y': (-0.0625, 0.0625)},
            scale=(0.9, 1.1),
            rotate=(-45, 45),
            border_mode=cv2.BORDER_CONSTANT,
            p=0.5
        ),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.2),

        # === Color / Illumination Augmentations ===
        A.ColorJitter(brightness=0.2, contrast=0.2,
                      saturation=0.2, hue=0.1, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(std_range=(0.02, 0.05), p=0.2),

        # === Normalization (ImageNet stats for pretrained backbones) ===
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transforms(img_size=512):
    """
    Validation/Test transform — NO augmentation, only resize + normalize.
    We must evaluate on clean, unaugmented images for fair comparison.
    """
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


In [ ]:
"""
Retinal DR Detection — Lesion Evaluation Metrics
==================================================
Per-channel and overall metrics for multi-label lesion segmentation.

For each of the 5 lesion types we compute:
  - Dice Coefficient  (primary metric)
  - IoU / Jaccard Index
  - Sensitivity (Recall)  — "did we find all lesion pixels?"
  - Specificity            — "did we avoid false alarms?"
  - Precision (PPV)

Then we report both per-type scores AND macro-averaged scores.
"""

import numpy as np
import torch
from sklearn.metrics import roc_auc_score, average_precision_score




def evaluate_multilabel_batch(pred_logits, targets, threshold=0.5):
    """
    Compute metrics for a batch of multi-label predictions.

    Args:
        pred_logits : (B, C, H, W)  raw model output BEFORE sigmoid
        targets     : (B, C, H, W)  binary ground truth
        threshold   : Binarization threshold for sigmoid output

    Returns:
        dict with per-channel and mean metrics
        Example: {'dice_MA': 0.72, ..., 'dice_mean': 0.68, ...}
    """
    smooth = 1e-6
    with torch.no_grad():
        pred_probs  = torch.sigmoid(pred_logits)
        pred_binary = (pred_probs > threshold).float()

        B, C, H, W = pred_binary.shape
        metrics = {}

        for ch in range(C):
            p = pred_binary[:, ch, :, :].contiguous().view(-1)
            t = targets[:, ch, :, :].contiguous().view(-1)

            tp = (p * t).sum()
            fp = (p * (1 - t)).sum()
            fn = ((1 - p) * t).sum()
            tn = ((1 - p) * (1 - t)).sum()

            dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
            iou  = (tp + smooth) / (tp + fp + fn + smooth)
            sens = (tp + smooth) / (tp + fn + smooth)
            spec = (tn + smooth) / (tn + fp + smooth)
            prec = (tp + smooth) / (tp + fp + smooth)

            lt = LESION_TYPES[ch] if ch < len(LESION_TYPES) else f'ch{ch}'
            metrics[f'dice_{lt}'] = dice.item()
            metrics[f'iou_{lt}']  = iou.item()
            metrics[f'sens_{lt}'] = sens.item()
            metrics[f'spec_{lt}'] = spec.item()
            metrics[f'prec_{lt}'] = prec.item()

        # Macro averages
        for m in ['dice', 'iou', 'sens', 'spec', 'prec']:
            vals = [metrics[f'{m}_{LESION_TYPES[c]}'] for c in range(C)]
            metrics[f'{m}_mean'] = np.mean(vals)

    return metrics


def evaluate_multilabel_full(pred_probs_all, targets_all,
                             num_classes=None, threshold=0.5):
    """
    Compute ALL metrics on the full test set (after collecting
    all predictions across batches).

    Args:
        pred_probs_all : np.array (N, C, H, W)  sigmoid probabilities
        targets_all    : np.array (N, C, H, W)  binary ground truth
        num_classes    : Number of channels (default: len(LESION_TYPES))

    Returns:
        dict with per-channel AND macro-averaged metrics including AUC
    """
    smooth = 1e-6
    if num_classes is None:
        num_classes = len(LESION_TYPES)

    pred_binary = (pred_probs_all > threshold).astype(np.float32)
    results = {}

    for ch in range(num_classes):
        p_flat = pred_binary[:, ch, :, :].ravel()
        t_flat = targets_all[:, ch, :, :].ravel()
        prob_flat = pred_probs_all[:, ch, :, :].ravel()

        tp = (p_flat * t_flat).sum()
        fp = (p_flat * (1 - t_flat)).sum()
        fn = ((1 - p_flat) * t_flat).sum()
        tn = ((1 - p_flat) * (1 - t_flat)).sum()

        dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
        iou  = (tp + smooth) / (tp + fp + fn + smooth)
        sens = (tp + smooth) / (tp + fn + smooth)
        spec = (tn + smooth) / (tn + fp + smooth)
        prec = (tp + smooth) / (tp + fp + smooth)
        acc  = (tp + tn) / (tp + tn + fp + fn + smooth)

        # AUC metrics (may fail if all labels are the same)
        try:
            auc_roc = roc_auc_score(t_flat, prob_flat)
        except ValueError:
            auc_roc = 0.0
        try:
            auc_pr = average_precision_score(t_flat, prob_flat)
        except ValueError:
            auc_pr = 0.0

        lt = LESION_TYPES[ch] if ch < len(LESION_TYPES) else f'ch{ch}'
        results[f'dice_{lt}']    = float(dice)
        results[f'iou_{lt}']     = float(iou)
        results[f'sens_{lt}']    = float(sens)
        results[f'spec_{lt}']    = float(spec)
        results[f'prec_{lt}']    = float(prec)
        results[f'acc_{lt}']     = float(acc)
        results[f'auc_roc_{lt}'] = float(auc_roc)
        results[f'auc_pr_{lt}']  = float(auc_pr)

    # Macro averages
    for m in ['dice', 'iou', 'sens', 'spec', 'prec', 'acc',
              'auc_roc', 'auc_pr']:
        vals = [results[f'{m}_{LESION_TYPES[c]}'] for c in range(num_classes)]
        results[f'{m}_mean'] = float(np.mean(vals))

    return results


## 2. Configuration

To answer your question: **Do we need the dataloader to train YOLO?**
**Answer: NO!** YOLO natively handles reading the images and parsing the dataset directly from the `data.yaml` config file. We don't use PyTorch loaders until the *evaluation* phase.


In [ ]:
import zipfile


zip_path = "yolo_lesion_dataset.zip"     # Change this to your actual zip file name
extract_dir = "dataset"           # Where to extract your files

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [ ]:
import zipfile


zip_path = "DR_dataset_processed.zip"     # Change this to your actual zip file name
extract_dir = "dataset2"           # Where to extract your files

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [ ]:
# Paths to your YOLO dataset directory
# (Update this if you moved the zip file on Kaggle or Colab)
YAML_PATH = 'dataset/data.yaml'

# Base paths for evaluation
DATA_DIR = 'dataset2'
TEST_CSV = os.path.join(DATA_DIR, 'test_split.csv')


## 3. Train YOLOv11


In [ ]:
# Load the pre-trained 'Small' Segmentation Model
model = YOLO('yolo11s-seg.pt')

# Train the model
# It will automatically save checkpoints inside 'runs/segment/train/' or whatever project name we set
results = model.train(
    data=YAML_PATH,
    epochs=100,
    imgsz=512,
    batch=8,
    patience=20,     # Stop early if no improvement
    device=0,        # Use CUDA
    project='lesion_results',
    name='YOLOv11-seg',

    # YOLO magic configurations for tiny dots handling
    cls=1.5,         # Force the loss to care more about class accuracy
    box=7.5,         # Force precise boundaries
    mosaic=0.5,      # Helps with finding tiny anomalies
    scale=0.5
)
print("✅ YOLO Training Complete!")


## 4. Evaluation (Apples-to-Apples with U-Net)

YOLO outputs objects (Bounding Boxes + Polygon Masks). Our `metrics.py` expects a clean `(B, 5, 512, 512)` probability tensor. Here, we write a **Reverse Conversion** script. We feed raw images into YOLO, take its polygons, and mathematically crush them back into the 5-channel PyTorch format so we can calculate Dice/IoU perfectly fairly.


In [ ]:
# Load the BEST trained YOLO model from the outputs
best_model_path = 'runs/segment/lesion_results/YOLOv11-seg2/weights/best.pt'
if not os.path.exists(best_model_path):
    print("WARNING: Could not find best model. Did training finish?")
else:
    trained_yolo = YOLO(best_model_path)

    # Initialize PyTorch Dataset JUST so we can load the exact same Ground Truth masks
    test_dataset = LesionDataset(TEST_CSV, DATA_DIR, transform=get_val_transforms(512), lesion_types=LESION_TYPES)

    df_test = pd.read_csv(TEST_CSV)
    df_test = df_test[df_test['has_lesion'] == True].reset_index(drop=True)

    all_preds = []
    all_targets = []

    for idx, row in tqdm(df_test.iterrows(), total=len(df_test), desc='Converting YOLO to PyTorch format'):
        img_path = os.path.join(DATA_DIR, row['img_path'])

        # 1. Ask YOLO to predict on the raw image!
        # (retina_masks=True forces high-resolution 512x512 mask outputs)
        res = trained_yolo.predict(img_path, imgsz=512, verbose=False, retina_masks=True)[0]

        # 2. Reconstruct the (5, 512, 512) tensor
        pred_tensor = np.zeros((5, 512, 512), dtype=np.float32)

        if res.masks is not None:
            masks_data = res.masks.data.cpu().numpy() # Shape: (Number of objects found, 512, 512)
            classes = res.boxes.cls.cpu().numpy().astype(int) # Shape: (Number of objects,)

            # Combine multiple objects of the same class into a single channel
            for inst_idx, cls_id in enumerate(classes):
                if cls_id < 5: # Ensure we only parse our 5 expected classes
                    pred_tensor[cls_id] = np.maximum(pred_tensor[cls_id], masks_data[inst_idx])

        all_preds.append(pred_tensor)

        # 3. Get the exact mathematically identical Ground Truth from PyTorch
        _, gt_mask = test_dataset[idx]
        all_targets.append(gt_mask.numpy())

    # 4. Final Evaluation
    all_preds = np.stack(all_preds, axis=0)
    all_targets = np.stack(all_targets, axis=0)

    print(f'Predictions shape: {all_preds.shape}')  # (N, 5, 512, 512)

    test_results = evaluate_multilabel_full(all_preds, all_targets)

    print(f'\n{"="*50}')
    print('  TEST RESULTS — YOLOv11-seg')
    print(f'{"="*50}')
    for lt in LESION_TYPES:
        print(f'\n  {lt}:')
        print(f'    Dice: {test_results[f"dice_{lt}"]:.4f}  |  IoU: {test_results[f"iou_{lt}"]:.4f}')
        print(f'    Sens: {test_results[f"sens_{lt}"]:.4f}  |  Spec: {test_results[f"spec_{lt}"]:.4f}')
        print(f'    Prec: {test_results[f"prec_{lt}"]:.4f}  |  AUC-ROC: {test_results[f"auc_roc_{lt}"]:.4f}')

    print(f'\n  MACRO AVERAGE:')
    print(f'    Dice: {test_results["dice_mean"]:.4f}  |  IoU: {test_results["iou_mean"]:.4f}')
    print(f'    Sens: {test_results["sens_mean"]:.4f}  |  Spec: {test_results["spec_mean"]:.4f}')
    print(f'{"="*50}')


## 5. Visualizer

Let's see the polygons placed over the actual image using OpenCV/YOLO's built-in plotting.


In [ ]:
if os.path.exists(best_model_path):
    # Just show the first 3 images visually
    sample_images = df_test['img_path'].tolist()[:3]

    for img_p in sample_images:
        full_path = os.path.join(DATA_DIR, img_p)

        # Run prediction
        res = trained_yolo.predict(full_path, imgsz=512, verbose=False)[0]

        # Ask YOLO to plot its boxes and masks onto the image array
        annotated_img = res.plot()

        # Convert BGR to RGB for matplotlib
        annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(8,8))
        plt.imshow(annotated_img)
        plt.axis('off')
        plt.title(f"YOLO Predictions: {os.path.basename(img_p)}")
        plt.show()


In [ ]:
def load_image_for_vis(idx):
    img_path = os.path.join(DATA_DIR, df_test.iloc[idx]['img_path'])
    # Load as RGB
    img = cv2.imread(img_path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

num_samples = 3
lesion_colors = {
    'MA': [1, 0, 0],     # Red
    'HE': [1, 0.5, 0],   # Orange
    'EX': [1, 1, 0],     # Yellow
    'OD': [0, 0.5, 1],   # Blue
    'CW': [1, 0, 1],     # Magenta
}

# We already have all_preds and all_targets from Cell 4!
fig, axes = plt.subplots(num_samples * 2, 2 + NUM_LESION_CLASSES,
                         figsize=(4 * (2 + NUM_LESION_CLASSES), 4 * num_samples * 2))
fig.suptitle('YOLOv11-seg — Ground Truth vs Predictions', fontsize=18, fontweight='bold', y=0.99)

for shown in range(num_samples):
    img_np = load_image_for_vis(shown)
    # Normalize image to 0-1 for plotting if it isn't already
    img_np = img_np / 255.0

    r_gt = shown * 2       # Row for Ground Truth
    r_pr = shown * 2 + 1   # Row for Prediction

    # --- Column 0: Original Image ---
    axes[r_gt, 0].imshow(img_np)
    axes[r_gt, 0].set_title('Image' if shown == 0 else '')
    axes[r_gt, 0].axis('off')

    axes[r_pr, 0].imshow(img_np)
    axes[r_pr, 0].axis('off')

    # --- Column 1: All combined ---
    # 1. Ground Truth All
    gt_overlay = img_np.copy()
    for ch, lt in enumerate(LESION_TYPES):
        gt_mask = all_targets[shown, ch] > 0.5
        gt_overlay[gt_mask] = lesion_colors[lt]
    axes[r_gt, 1].imshow(gt_overlay)
    axes[r_gt, 1].set_title('Ground Truth (All)' if shown == 0 else '')
    axes[r_gt, 1].axis('off')

    # 2. Prediction All
    pr_overlay = img_np.copy()
    for ch, lt in enumerate(LESION_TYPES):
        # We don't need a threshold here; YOLO Masks are already calculated correctly
        pred_mask = all_preds[shown, ch] > 0.5
        pr_overlay[pred_mask] = lesion_colors[lt]
    axes[r_pr, 1].imshow(pr_overlay)
    axes[r_pr, 1].set_title('Prediction (All)' if shown == 0 else '')
    axes[r_pr, 1].axis('off')

    # --- Columns 2-6: Individual Lesions ---
    for ch, lt in enumerate(LESION_TYPES):
        gt_mask = all_targets[shown, ch]
        pr_mask = (all_preds[shown, ch] > 0.5).astype(float)

        # Plot purely black-and-white mask for Ground Truth
        axes[r_gt, 2 + ch].imshow(gt_mask, cmap='gray')
        axes[r_gt, 2 + ch].set_title(f'{lt} (Real)' if shown == 0 else '')
        axes[r_gt, 2 + ch].axis('off')

        # Plot purely black-and-white mask for Prediction
        axes[r_pr, 2 + ch].imshow(pr_mask, cmap='gray')
        axes[r_pr, 2 + ch].set_title(f'{lt} (Pred)' if shown == 0 else '')
        axes[r_pr, 2 + ch].axis('off')

plt.tight_layout()
plt.show()
print('✅ YOLO Prediction visualization complete! Remember to check the "runs" folder for the data charts!')


In [ ]:
import json
import os

# Google Colab specific download library
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 1. Save the Test Results JSON
save_dir = 'lesion_results/YOLOv11-seg'
os.makedirs(save_dir, exist_ok=True)

results_file = os.path.join(save_dir, 'YOLOv11-seg_results.json')

# Convert any weird numpy numbers to pure Python floats so JSON doesn't crash
clean_results = {k: float(v) for k, v in test_results.items()}

with open(results_file, 'w') as f:
    json.dump(clean_results, f, indent=4)

print(f"✅ Metrics saved to {results_file}")


# 2. Save your side-by-side Visualizations
vis_file = os.path.join(save_dir, 'YOLOv11-seg_predictions.png')
# Assuming 'fig' is still loaded in memory from the previous cell
fig.savefig(vis_file, dpi=150, bbox_inches='tight')
print(f"✅ Visualizations saved to {vis_file}")


# 3. Automatically Trigger Browser Downloads
if IN_COLAB:
    print("⏳ Triggering downloads in your browser...")
    files.download(results_file)
    files.download(vis_file)

    # Try to download YOLO's cool built-in charts too!
    try:
        files.download('runs/segment/lesion_results/YOLOv11-seg/confusion_matrix.png')
        files.download('runs/segment/lesion_results/YOLOv11-seg/results.png')
    except:
        print("Could not find YOLO's built-in charts to download (this is fine).")
else:
    print("Since you are running locally, check your 'lesion_results/YOLOv11-seg/' folder for everything!")
